In [1]:
import os
import numpy as np
import pandas as pd
from collections import OrderedDict

In [2]:
def choose_best_combination(dict_DF):
    max_dict = {}
    for k in dict_DF:
        max_dict[k] = max(transform_df_to_plot_format(dict_DF[k])['mean'])
    max_key = max(max_dict, key=max_dict.get)
    return max_key

def obtain_max_value(dict_DF):
    max_dict = {}
    for k in dict_DF:
        max_dict[k] = max(transform_df_to_plot_format(dict_DF[k])['mean'])
    max_value = max(max_dict.values())
    return max_value
    
def obtain_min_value(dict_DF):
    min_dict = {}
    for k in dict_DF:
        min_dict[k] = min(transform_df_to_plot_format(dict_DF[k])['mean'])
    min_value = min(min_dict.values())
    return min_value

def compute_q_mean(x):
    return np.mean(x)
def compute_std_plus(x):
    return np.mean(x)+np.std(x)
def compute_std_minus(x):
    return np.mean(x)-np.std(x)

def transform_df_to_plot_format(df):
    
    grouped_df = df.groupby("time")
    grouped_lists = grouped_df["scaled_eu"].apply(list)
    grouped_lists = grouped_lists.reset_index()
    grouped_lists['std_minus'] = grouped_lists['scaled_eu'].apply(compute_std_minus)
    grouped_lists['std_plus'] = grouped_lists['scaled_eu'].apply(compute_std_plus)
    grouped_lists['mean'] = grouped_lists['scaled_eu'].apply(compute_q_mean)
    grouped_lists['|Q|'] = df['|Q|'].values[0]
    grouped_lists['|X|'] = df['|X|'].values[0]
    grouped_lists['|T|'] = df['|T|'].values[0]
    grouped_lists['rho'] = df['rho'].values[0]
    grouped_lists['k'] = df['k'].values[0]
    grouped_lists['solver'] = df['solver'].values[0]
    grouped_lists['attacker'] = df['attacker'].values[0]
    return grouped_lists
            
def create_dict_DF_APS(exp, att_str, comb_num):
    dict_DF = OrderedDict()
    exp_dir = 'experiment_'+str(exp)+'_scaled_csv/'
    for hyp_c in ['cs_1', 'cs_500']:
        for f in os.listdir(exp_dir + '/' +'APS'+ '/'+ att_str+'/' + hyp_c):
            if '.json' in f:
                if 'comb_'+str(comb_num) in f:
                    if hyp_c == 'cs_1':
                        dict_DF['A'] = pd.read_json(exp_dir + '/' +'APS' + '/'+ att_str+'/' + hyp_c +'/'+f ,orient = 'split')
                    elif hyp_c == 'cs_500':
                        dict_DF['B'] = pd.read_json(exp_dir + '/' +'APS' + '/'+ att_str+'/' + hyp_c +'/'+f ,orient = 'split')
    return dict_DF
    
def create_dict_DF_RS_SA(exp,  solver, att_str, comb_num):
    dict_DF = OrderedDict()
    exp_dir = 'experiment_'+str(exp)+'_scaled_csv/'
    #for hyp_c in os.listdir(exp_dir + '/' +solver + '/'+ att_str):
    for hyp_c in ['A', 'B', 'C', 'D','E', 'F']:
        for f in os.listdir(exp_dir + '/' +solver + '/'+ att_str+'/' + hyp_c):
            if '.json' in f:
                if 'comb_'+str(comb_num) in f:
                    if hyp_c == 'A':
                        dict_DF['A'] = pd.read_json(exp_dir + '/' +solver + '/'+ att_str+'/' + hyp_c +'/'+f ,orient = 'split')
                    elif hyp_c == 'B':
                        dict_DF['B'] = pd.read_json(exp_dir + '/' +solver + '/'+ att_str+'/' + hyp_c +'/'+f ,orient = 'split')
                    elif hyp_c == 'C':
                        dict_DF['C'] = pd.read_json(exp_dir + '/' +solver + '/'+ att_str+'/' + hyp_c +'/'+f ,orient = 'split')
                    elif hyp_c == 'D':
                        dict_DF['D'] = pd.read_json(exp_dir + '/' +solver + '/'+ att_str+'/' + hyp_c +'/'+f ,orient = 'split')
                    elif hyp_c == 'E' :
                        dict_DF['E'] = pd.read_json(exp_dir + '/' +solver + '/'+ att_str+'/' + hyp_c +'/'+f ,orient = 'split')
                    elif hyp_c == 'F':
                        dict_DF['F'] = pd.read_json(exp_dir + '/' +solver + '/'+ att_str+'/' + hyp_c +'/'+f ,orient = 'split')
    return dict_DF


def create_dict_DF_solver(exp_, solver_, att_str_, comb_num_):
    
    if solver_ == 'APS':
        dict_DF = create_dict_DF_APS(exp = exp_, att_str = att_str_, comb_num = comb_num_)
    elif solver_ == 'MCTS':
        dict_DF = create_dict_DF_RS_SA(exp = exp_, solver = solver_, att_str = att_str_, comb_num = comb_num_)
    elif solver_ == 'SA':
        dict_DF = create_dict_DF_RS_SA(exp = exp_, solver = solver_, att_str = att_str_, comb_num = comb_num_)
    
    return dict_DF

def obtain_best_combination_loop(exp_list, solver_list, att_list, comb_list):
    ld = []
    for exp in exp_list:
        for solver in solver_list:
            for att in att_list:
                for comb in comb_list:
                    dict_DF = create_dict_DF_solver(exp_ = exp , solver_ = solver , att_str_ = att, comb_num_ = comb)
                    b_hc = choose_best_combination(dict_DF)
                    ld.append({'exp':exp,
                               'solver':solver,
                                'att': att,
                                'comb_num': comb,
                                'best_hyp': b_hc})
    return pd.DataFrame(ld)


In [ ]:
df_wrong = pd.read_csv('best_combination_df_appendix_wrong.csv')

In [3]:
df_def_right = obtain_best_combination_loop(exp_list = ['2','3'], 
                             solver_list = ['APS','MCTS','SA'], 
                             att_list = ['attr', 'rep', 'dd', 'pd'], 
                             comb_list = ['1','2','3','4'])

In [4]:
df_def_right.to_csv('definitive_best_combination_loop_all_experiments.csv', index  = False)